In [0]:
"""
베딩 모델은 배포에 시간이 걸려서, 관리자가 (내가) 배포해둘 것
우측 자기 이름 -> 설정 -> developer -> 새 토큰 생성 -> 복사
"""
DATABRICKS_TOKEN = 'REDACTED'

In [0]:
# Test

import requests
import json
import pandas as pd


test_pd = pd.DataFrame({
    "input": ["Samsung ETF overview", "KOSPI ETF factsheet"]
})

url = "https://adb-7405614771934365.5.azuredatabricks.net/serving-endpoints/bge_m3/invocations"
headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}
payload = {
    "dataframe_split": test_pd.to_dict(orient="split")
}

resp = requests.post(url, headers=headers, data=json.dumps(payload))
print(resp.status_code)
print(resp.text[:2000])

In [0]:
import requests
import json
import pandas as pd

def score_model(dataset: pd.DataFrame):
    if not isinstance(dataset, pd.DataFrame):
        raise ValueError("dataset must be a pandas DataFrame")

    # 모델이 기대하는 컬럼명은 반드시 input
    if "input" in dataset.columns:
        scoring_df = dataset[["input"]].copy()
    elif "text" in dataset.columns:
        scoring_df = dataset[["text"]].rename(columns={"text": "input"}).copy()
    else:
        raise ValueError(f"dataset must contain either 'input' or 'text' column. got: {list(dataset.columns)}")

    scoring_df["input"] = scoring_df["input"].fillna("").astype(str)

    url = "https://adb-7405614771934365.5.azuredatabricks.net/serving-endpoints/bge_m3/invocations"
    headers = {
        "Authorization": f"Bearer {DATABRICKS_TOKEN}",
        "Content-Type": "application/json"
    }

    payload = {
        "dataframe_split": scoring_df.to_dict(orient="split")
    }

    response = requests.post(
        url=url,
        headers=headers,
        data=json.dumps(payload, allow_nan=True)
    )

    print("status_code:", response.status_code)
    print("payload columns:", payload["dataframe_split"]["columns"])
    print("response preview:", response.text[:1000])

    if response.status_code != 200:
        raise Exception(f"Request failed with status {response.status_code}, {response.text}")

    return response.json()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F
import pandas as pd

# ===== 설정 =====
source_table = "cat_adb_workshop.sch_user0_unstructured.pdf_chunks"
target_table = "cat_adb_workshop.sch_user0_unstructured.pdf_embedded"

batch_row_size = 200   # Spark → Pandas 단위
endpoint_batch_size = 16  # endpoint 호출 단위

# ===== 데이터 준비 =====
df = spark.table(source_table)

src = (
    df
    .filter(F.col("text").isNotNull() & (F.length("text") > 0))
    .withColumn("rn", F.row_number().over(Window.orderBy("source_file", "page_num")))
)

total_rows = src.count()
print("total_rows:", total_rows)

# ===== 전체 데이터 임베딩 =====
for start in range(1, total_rows + 1, batch_row_size):
    end = start + batch_row_size - 1
    print(f"\nProcessing rows {start} ~ {end}")

    # 1. Spark → Pandas
    batch_pd = (
        src
        .filter((F.col("rn") >= start) & (F.col("rn") <= end))
        .drop("rn")
        .toPandas()
    )

    # 2. endpoint용 컬럼 변환 (핵심)
    scoring_pd = pd.DataFrame({
        "input": batch_pd["text"].fillna("").astype(str)
    })

    # 3. 임베딩 호출
    resp = score_model(scoring_pd)

    # 4. embedding 추출
    if "predictions" in resp:
        embeddings = resp["predictions"]
    elif "data" in resp:
        embeddings = resp["data"]
    elif "outputs" in resp:
        embeddings = resp["outputs"]
    else:
        raise ValueError(f"Unknown response format: {resp}")

    if len(embeddings) != len(batch_pd):
        raise ValueError("embedding size mismatch")

    # 5. 원본에 붙이기
    batch_pd["embedding"] = embeddings

    # 6. float 변환 (중요)
    batch_pd["embedding"] = batch_pd["embedding"].apply(
        lambda x: [float(v) for v in x]
    )

    # 7. Spark 변환
    embedded_sdf = spark.createDataFrame(batch_pd)

    # 8. 저장 (append)
    (
        embedded_sdf
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(target_table)
    )

    print(f"Saved rows {start} ~ {end}")